# ML Engine for Linux Scheduler
## Step 2: Model Training, Evaluation & Comparison

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from src.train import (get_all_models, get_supervised_models, train_model,
                       train_all_models, save_model, load_model, prepare_data)
from src.evaluate import (evaluate_model, evaluate_all_models, evaluate_kmeans,
                          get_feature_importance, generate_comparison_report,
                          get_classification_report_text)
from src.visualize import (plot_confusion_matrix, plot_feature_importance,
                           plot_model_comparison)
from src.pipeline import simulate_realtime_scheduler, compare_ml_vs_rules
from src import LABEL_MAP, LABEL_NAMES, TARGET_COLUMN, get_all_feature_columns

## 2.1 Load Processed Data

In [ ]:
PROCESSED_PATH = os.path.join('data', 'processed', 'clean_metrics.csv')
df = pd.read_csv(PROCESSED_PATH)
print(f"Loaded dataset: {df.shape}")
df.head()

## 2.2 Prepare Train/Test Split
Following the paper: 70% training, 30% testing with stratified sampling.

In [ ]:
X_train, X_test, y_train, y_test = prepare_data(df, test_size=0.30, random_state=42)
print(f"\nTraining set class distribution:")
print(y_train.value_counts().sort_index())
print(f"\nTest set class distribution:")
print(y_test.value_counts().sort_index())

## 2.3 Train All Models
Training 5 supervised classification models as specified in the paper:
1. Decision Tree
2. K-Nearest Neighbors (KNN)
3. Random Forest
4. **Gradient Boosting** (Primary model)
5. Logistic Regression

In [ ]:
trained_models = train_all_models(X_train, y_train, include_kmeans=False)
print(f"\nSuccessfully trained {len(trained_models)} models")

## 2.4 Evaluate All Models

In [ ]:
results_df = evaluate_all_models(trained_models, X_test, y_test)
print("\n" + "="*70)
print("MODEL COMPARISON TABLE")
print("="*70)
print(results_df.to_string(index=False))

## 2.5 Model Comparison Chart

In [ ]:
FIGURES_DIR = os.path.join('output', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

plot_model_comparison(results_df, metric='accuracy', save_path=os.path.join(FIGURES_DIR, 'model_comparison_accuracy.png'))

In [ ]:
plot_model_comparison(results_df, metric='f1_score', save_path=os.path.join(FIGURES_DIR, 'model_comparison_f1.png'))

## 2.6 Detailed Analysis: Best Model
The paper recommends Gradient Boosting for its ability to capture complex relationships.

In [ ]:
# Select best model by accuracy
best_model_name = results_df.iloc[0]['model_name']
best_model = trained_models[best_model_name]
print(f"Best performing model: {best_model_name}")
print(f"Accuracy: {results_df.iloc[0]['accuracy']:.4f}")
print(f"\nDetailed Classification Report:")
print(get_classification_report_text(best_model, X_test, y_test))

### Confusion Matrix - Best Model

In [ ]:
from sklearn.metrics import confusion_matrix
y_pred = best_model.predict(X_test)
plot_confusion_matrix(y_test, y_pred,
                      title=f'Confusion Matrix - {best_model_name}',
                      save_path=os.path.join(FIGURES_DIR, 'confusion_matrix_best.png'))

### Confusion Matrix - Gradient Boosting (Paper's Primary Model)

In [ ]:
gb_model = trained_models['Gradient Boosting']
y_pred_gb = gb_model.predict(X_test)
plot_confusion_matrix(y_test, y_pred_gb,
                      title='Confusion Matrix - Gradient Boosting',
                      save_path=os.path.join(FIGURES_DIR, 'confusion_matrix_gradient_boosting.png'))

### Feature Importance

In [ ]:
feature_names = get_all_feature_columns()

# Feature importance for Gradient Boosting
importances = get_feature_importance(gb_model, feature_names)
if importances is not None:
    print("Gradient Boosting Feature Importance:")
    print(importances)
    plot_feature_importance(importances.values, importances.index.tolist(),
                            title='Feature Importance - Gradient Boosting',
                            save_path=os.path.join(FIGURES_DIR, 'feature_importance_gb.png'))

In [ ]:
# Feature importance for Random Forest
rf_model = trained_models['Random Forest']
importances_rf = get_feature_importance(rf_model, feature_names)
if importances_rf is not None:
    print("Random Forest Feature Importance:")
    print(importances_rf)
    plot_feature_importance(importances_rf.values, importances_rf.index.tolist(),
                            title='Feature Importance - Random Forest',
                            save_path=os.path.join(FIGURES_DIR, 'feature_importance_rf.png'))

## 2.7 K-Means Clustering Analysis
Unsupervised analysis to discover natural process groupings.

In [ ]:
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(X_train)

kmeans_results = evaluate_kmeans(kmeans, X_test, y_test)
print("\nK-Means Cluster Centers:")
centers_df = pd.DataFrame(kmeans.cluster_centers_, columns=feature_names)
print(centers_df.round(2))

## 2.8 Save Best Model

In [ ]:
MODEL_DIR = os.path.join('models')
os.makedirs(MODEL_DIR, exist_ok=True)

# Save Gradient Boosting (paper's primary model)
save_model(gb_model, os.path.join(MODEL_DIR, 'gradient_boosting_scheduler.pkl'))
print(f"Saved Gradient Boosting model")

# Also save best model if different
if best_model_name != 'Gradient Boosting':
    save_model(best_model, os.path.join(MODEL_DIR, f'{best_model_name.lower().replace(" ", "_")}_scheduler.pkl'))
    print(f"Saved {best_model_name} model")

# Save comparison results
results_path = generate_comparison_report(results_df)
print(f"\nResults saved to: {results_path}")

## 2.9 Real-Time Inference Simulation
Demonstrate the trained model predicting scheduling decisions for new processes.

In [ ]:
print("=" * 60)
print("REAL-TIME SCHEDULER SIMULATION")
print("=" * 60)
result_df = simulate_realtime_scheduler(gb_model, n_processes=10)

## 2.10 ML vs Rule-Based Comparison

In [ ]:
print("=" * 60)
print("ML MODEL vs RULE-BASED COMPARISON")
print("=" * 60)
comparison = compare_ml_vs_rules(gb_model, n_processes=15)

## Summary

### Key Findings:
- Trained and compared 5 supervised ML models + K-Means clustering
- Gradient Boosting is the paper's recommended model
- The ML model can predict process scheduling priorities accurately
- Feature importance analysis reveals which process attributes matter most

### Three Schedule Classes:
| Class | Label | Description |
|-------|-------|-------------|
| 0 | IMMEDIATELY SCHEDULE | High-priority/CPU-intensive processes |
| 1 | NEXT SCHEDULE | Normal interactive processes |
| 2 | LATELY SCHEDULE | Background/idle processes |